# 07. Auto-Labeling — 파인튜닝 모델로 YOLO 데이터셋 만들기

`peanut_yolov8`으로 학습한 모델(`best.pt`)을 이용해 `20260702_bad` 폴더의 이미지에서 땅콩 위치(bounding box)를 자동으로 찾아내고, YOLO 학습용 데이터셋(`images/` + `labels/` + `data.yaml`)을 만듭니다.

**목표**: 품질(good/bad) 분류가 아니라 **땅콩 형태(위치)를 최대한 다 검출**하는 것입니다.
- 이 폴더의 땅콩은 전부 bad 등급이므로, 검출된 모든 박스는 모델의 예측 클래스와 무관하게 **`1 (bad)`로 강제 기록**합니다.
- confidence는 아주 낮게(`0.02`) 잡아서 웬만한 땅콩은 다 박스가 쳐지도록 합니다.

절차: **① 이미지 1장으로 확인 → ② 라벨 변환 검증 → ③ 전체 배치 처리 → ④ 결과 시각화**

## 0. 환경 세팅

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
from pathlib import Path
from ultralytics import YOLO

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", DEVICE)

MODEL_PATH = "peanut_yolov8/weights/best.pt"


SRC_DIR = Path("20260702_good")
OUT_DIR = Path("20260702_auto_labeled_good")

model = YOLO(MODEL_PATH)
print("model classes:", model.names)   # {0: 'good', 1: 'bad'} — 참고용, 실제 라벨은 전부 bad(1)로 강제

TARGET_CLASS_ID = 0

# 자동 라벨링 파라미터
CONF = 0.02      # 낮게 잡아 웬만한 땅콩은 다 박스가 쳐지도록 함 (품질 분류가 아니라 형태 검출이 목적)
IOU = 0.3        # NMS 임계값 — 겹치는 중복 박스 제거 => 30% 겹치는 정도만 허용
IMGSZ = 800      # 학습 시 imgsz(800)와 동일하게 맞춤

img_files = sorted(SRC_DIR.glob("*.png"))
print("총 이미지 수:", len(img_files))

## 1. 단일 이미지로 먼저 확인

전체를 돌리기 전에, conf=0.02 설정이 실제로 땅콩을 잘 잡는지 한 장으로 눈으로 확인합니다.

In [ ]:
from PIL import Image

test_img = img_files[101]   # 땅콩이 잘 보이는 편인 임의의 한 장
print("테스트 이미지:", test_img.name)

r = model.predict(
    str(test_img),
    device=DEVICE,
    conf=CONF,
    iou=IOU,
    agnostic_nms=True,   # good/bad 두 클래스가 같은 땅콩에 중복으로 박스 치는 것 방지
    imgsz=IMGSZ,
    verbose=False,
)[0]

print("탐지 개수:", len(r.boxes))
im_bgr = r.plot(line_width=2)
Image.fromarray(im_bgr[..., ::-1])   # BGR -> RGB

## 2. 라벨(.txt) 변환 검증

탐지된 박스를 YOLO 라벨 포맷(`class cx cy w h`, 0~1 정규화)으로 변환합니다.
클래스는 모델의 예측(good/bad)을 무시하고 **전부 `1 (bad)`**로 기록합니다.

In [ ]:
def boxes_to_yolo_lines(boxes, class_id=TARGET_CLASS_ID):
    lines = []
    for xywhn in boxes.xywhn.tolist():
        cx, cy, w, h = xywhn
        lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return lines

lines = boxes_to_yolo_lines(r.boxes)
print(f"{len(lines)}줄 생성")
print("\n".join(lines[:5]), "...")

왕복 검증 — 생성한 라벨을 원본 이미지 위에 다시 그려서 눈으로 확인합니다.

In [ ]:
from PIL import ImageDraw

def draw_yolo_labels(img_path, lines, color="lime"):
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    draw = ImageDraw.Draw(im)
    for line in lines:
        cid, cx, cy, w, h = map(float, line.split())
        x1 = (cx - w / 2) * W; y1 = (cy - h / 2) * H
        x2 = (cx + w / 2) * W; y2 = (cy + h / 2) * H
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
    return im

draw_yolo_labels(test_img, lines)

## 3. 출력 데이터셋 폴더 준비

`images/`, `labels/`, `data.yaml`로 구성된 YOLO 데이터셋 폴더를 만듭니다.
train/val 분리 없이 하나의 세트로 만들고, 나중에 학습 시 직접 분리합니다.

In [ ]:
import yaml

(OUT_DIR / "images").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "labels").mkdir(parents=True, exist_ok=True)

data_yaml = {
    "path": str(OUT_DIR.resolve()),
    "train": "images",
    "val": "images",   # train/val 분리 없이 하나의 세트 (추후 학습 시 직접 분리)
    "names": {0: "good", 1: "bad"},   # 원본 모델과 클래스 id 호환 (이 배치는 전부 bad만 존재)
}
with open(OUT_DIR / "data.yaml", "w") as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print((OUT_DIR / "data.yaml").read_text())

## 4. 전체 이미지 배치 처리

`20260702_bad`의 모든 이미지를 순회하며 이미지를 복사하고 라벨(.txt)을 생성합니다.
박스가 하나도 안 나온 이미지는 따로 목록으로 모아 나중에 검토합니다.

In [ ]:
import shutil

zero_box_images = []
total_boxes = 0

for i, img_path in enumerate(img_files, 1):
    r = model.predict(
        str(img_path),
        device=DEVICE,
        conf=CONF,
        iou=IOU,
        agnostic_nms=True,
        imgsz=IMGSZ,
        verbose=False,
    )[0]

    lines = boxes_to_yolo_lines(r.boxes)
    if not lines:
        zero_box_images.append(img_path.name)
    total_boxes += len(lines)

    shutil.copy(img_path, OUT_DIR / "images" / img_path.name)
    label_path = OUT_DIR / "labels" / (img_path.stem + ".txt")
    label_path.write_text("\n".join(lines))

    if i % 50 == 0 or i == len(img_files):
        print(f"[{i}/{len(img_files)}] 처리 완료")

print()
print("총 이미지:", len(img_files))
print("총 박스 수:", total_boxes)
print("박스 0개 이미지:", len(zero_box_images), "장")

In [ ]:
# 박스 0개 이미지 목록 — 실제로 땅콩이 없는 빈 프레임인지 확인 필요
print("\n".join(zero_box_images))

## 5. 결과 무작위 샘플 시각화

생성된 데이터셋에서 몇 장을 무작위로 뽑아 라벨이 잘 그려지는지 최종 확인합니다.

In [ ]:
import random
import matplotlib.pyplot as plt

random.seed(42)
candidates = [p.name for p in img_files if p.name not in zero_box_images]
sample_names = random.sample(candidates, 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, name in zip(axes.flat, sample_names):
    img_p = OUT_DIR / "images" / name
    lbl_p = OUT_DIR / "labels" / (Path(name).stem + ".txt")
    lines = lbl_p.read_text().strip().splitlines()
    im = draw_yolo_labels(img_p, lines)
    ax.imshow(im)
    ax.set_title(f"{name} ({len(lines)} boxes)", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 체크리스트

- [ ] `auto_labeled_bad/images/`, `labels/` 파일 개수가 원본 이미지 수와 일치하는가
- [ ] `data.yaml`의 `names`가 원본 모델(`{0: good, 1: bad}`)과 일치하는가
- [ ] 박스 0개 이미지들을 열어서 실제로 빈 프레임인지 확인했는가
- [ ] 무작위 샘플에서 라벨 박스가 땅콩 위치와 잘 맞는가
- [ ] (선택) 박스가 부정확하거나 놓친 경우 CVAT/Label Studio 등으로 수동 보정